# Sign Detection: video to review sheet to GIS file

This notebook turns a truck video into a spreadsheet of signs for a person to check, then turns the checked sheet into a CSV with lat/lon for GIS.

**How to use it**

1. **Run cells 1, 2 and 3 first** (every time you open the notebook). They connect to Drive and get the code. Cell 2 takes a few minutes the first time. **Please make sure to add the LADOT Sign Detection folder as a shortcut in MyDrive**
2. **To check a new video:** put the video in the `clips` folder, type its file name in cell 4, and run cell 4. When it finishes, it saves `review.xlsx` in the `reviews` folder.
3. **Review the sheet:** open `review.xlsx` from Drive (right-click, Open with, Google Sheets). For each row pick a **Decision** (correct, wrong name, not a sign, can't tell). If the name is wrong, pick the right one in **Correct name**. Leave a row blank if you have not reviewed it. Then download it as an .xlsx file (File, Download, Microsoft Excel) and upload it back to the same folder in `reviews`, replacing the old `review.xlsx`.
4. **Make the GIS file:** type the name of the review folder in cell 5 and run it. It saves `gis_export.csv` in the `exports` folder.

**Good to know**
- The model proposes and a person confirms. Only rows marked correct or wrong name (with a location) go into the GIS file. Everything else goes to `not_exported.csv` with a reason.
- Sign locations are estimates from GPS plus depth, so they can be off by several meters.
- If the folder was shared with you, add a shortcut to it in My Drive first (right-click the folder, Organize, Add shortcut). Otherwise cell 1 cannot find it.
- Long videos take a long time. Try a short clip first.



In [ ]:
#@title 1. Connect to Drive
FOLDER = "LADOT Sign Detection" #@param {type:"string"}

import os, shutil
from google.colab import drive
drive.mount("/content/drive")

BASE = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(BASE):
    raise SystemExit(f"Folder not found: {BASE}\nIf it was shared with you, add a shortcut to it in My Drive "
                     "(right-click the folder, Organize, Add shortcut), or fix the name above.")
for sub in ["reviews", "exports"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

needed = ["models/models_run_g", "config/label_config_G.xml", "config/sign_lookup.csv",
          "config/Elane-Merged Master Sign Catalog - Revised 2025.10.29.xlsx", "clips"]
missing = [n for n in needed if not os.path.exists(f"{BASE}/{n}")]
if missing:
    raise SystemExit("These are missing from the Drive folder: " + ", ".join(missing))
print("Drive folder OK:", BASE)
print("Clips in the folder:", [f for f in os.listdir(f"{BASE}/clips") if f.lower().endswith((".mp4", ".mkv", ".mov"))])


In [ ]:
#@title 2. Install tools (a few minutes the first time)
import os, shutil, subprocess

# ExifTool reads the GPS and camera settings from the video.
# The version in Ubuntu's package list can be old, so try the current one first.
!apt-get -qq install -y libimage-exiftool-perl > /dev/null 2>&1
!mkdir -p /opt/exiftool && curl -sSL https://github.com/exiftool/exiftool/archive/refs/tags/13.40.tar.gz | tar xz -C /opt/exiftool --strip-components=1 2>/dev/null
if os.path.exists("/opt/exiftool/exiftool"):
    os.environ["PATH"] = "/opt/exiftool:" + os.environ["PATH"]
print("ExifTool version:", subprocess.run(["exiftool", "-ver"], capture_output=True, text=True).stdout.strip())

!pip -q install ultralytics==8.3.228 transformers==4.57.1 pyproj==3.7.2 haversine==2.9.0 polyline==2.0.3 piexif==1.1.3 google-maps-routing==0.7.0 gspread==6.2.1 oauth2client==4.1.3 openpyxl
!pip install torchmetrics
print("Tools installed.")


In [ ]:
#@title 3. Get the code and model files
BRANCH = "master" #@param {type:"string"}

import os, shutil
os.chdir("/content")
!rm -rf /content/sign_pipeline
!git clone -q --branch {BRANCH} https://github.com/ElaneShane/Google-Virtual-Drive.git /content/sign_pipeline
os.chdir("/content/sign_pipeline")

# Files that are not in git come from the Drive folder.
os.makedirs("models", exist_ok=True)
os.makedirs("uncomitted", exist_ok=True)
shutil.copytree(f"{BASE}/models/models_run_g", "models/models_run_g", dirs_exist_ok=True)
shutil.copy(f"{BASE}/config/label_config_G.xml", "uncomitted/label_config_G.xml")
home_data = os.path.expanduser("~/ladot-virtual-drive")
os.makedirs(home_data, exist_ok=True)
MASTER_CATALOG = "Elane-Merged Master Sign Catalog - Revised 2025.10.29.xlsx"

shutil.copy(
    f"{BASE}/config/{MASTER_CATALOG}",
    f"{home_data}/{MASTER_CATALOG}"
)
shutil.copy(
    f"{BASE}/config/sign_lookup.csv",
    f"{home_data}/sign_lookup.csv"
)

print("Sign lookup copied:", os.path.exists(f"{home_data}/sign_lookup.csv"))

# Check the code loads, and show which version we got.
!git log --oneline -1
!python -c "import drive, helper, camera; print('code imports OK')"
print("Model files:", os.listdir("models/models_run_g"))


In [ ]:
#@title 4. Run a video and build the review sheet
CLIP = "W7thSt_SPacificAve_to_SPalosVerdesSt_SB.MP4" #@param {type:"string"}
SECONDS_BETWEEN_FRAMES = 2 #@param {type:"number"}

import glob, os, shutil, time

src = f"{BASE}/clips/{CLIP}"
if not os.path.exists(src):
    raise SystemExit(f"Clip not found: {src}")
os.makedirs("/content/local_clips", exist_ok=True)
local = f"/content/local_clips/{CLIP}"
t0 = time.time()
shutil.copy(src, local)                      # reading a big video straight from Drive is slow
print(f"Copied clip ({time.time() - t0:.0f} s)")

!python run_gopro_tests.py "{local}" models/models_run_g 1280 {SECONDS_BETWEEN_FRAMES}

stem = os.path.splitext(CLIP)[0].replace(" ", "_")
RUN = max(glob.glob(f"runs/{stem}_*"), key=os.path.getmtime)
print("Run folder:", RUN)

!python make_review.py "{RUN}" {SECONDS_BETWEEN_FRAMES}
!python make_review_xlsx.py "{RUN}"

out = f"{BASE}/reviews/{os.path.basename(RUN)}"
os.makedirs(out, exist_ok=True)

for f in ["review.xlsx", "review.csv", "debug_detections.csv", "signs.csv"]:
    if os.path.exists(f"{RUN}/{f}"):
        shutil.copy(f"{RUN}/{f}", out)

# Copy the original detection crops into the same Drive run folder
import csv
from pathlib import Path

run_path = Path(RUN)
drive_crops = Path(out) / "crops"
drive_crops.mkdir(parents=True, exist_ok=True)

review_csv = run_path / "review.csv"

copied = 0
missing = 0

with review_csv.open("r", newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)

    for row in reader:
        picture = (row.get("best_picture") or "").strip()

        if not picture:
            missing += 1
            continue

        source = Path(picture)

        if not source.is_absolute():
            source = run_path / source

        if not source.exists():
            print(f"Missing crop: {source}")
            missing += 1
            continue

        try:
            relative_path = source.relative_to(run_path)
        except ValueError:
            relative_path = Path(source.name)

        destination = drive_crops / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(source, destination)
        copied += 1

print(f"Copied {copied} original crops to: {drive_crops}")
print(f"Missing crop files: {missing}")

print("\nSaved to Drive:", out)
print("Next: open review.xlsx there, review it, and upload the finished .xlsx back to that same folder.")

### Review the sheet now

Open `review.xlsx` in the folder printed above. Fill in **Decision** (and **Correct name** when the name is wrong). Download it as an .xlsx file and upload it back into the same folder, replacing the old `review.xlsx`. Then run cell 5.

In [ ]:
#@title 5. Make the GIS file from the reviewed sheet
REVIEW_FOLDER = "W7thSt_SPacificAve_to_SPalosVerdesSt_SB_2026-11-18_185932" #@param {type:"string"}

import os, shutil
folder = f"{BASE}/reviews/{REVIEW_FOLDER}"
if not os.path.exists(f"{folder}/review.xlsx"):
    raise SystemExit(f"No review.xlsx in {folder}")

os.chdir("/content/sign_pipeline")
!python export_reviewed.py "{folder}"

os.makedirs(f"{BASE}/exports", exist_ok=True)
shutil.copy(f"{folder}/gis_export.csv", f"{BASE}/exports/{REVIEW_FOLDER}_gis_export.csv")
print("\nSaved:", f"{BASE}/exports/{REVIEW_FOLDER}_gis_export.csv")
print("Rows that were not exported, and why, are in", f"{folder}/not_exported.csv")
